# Fredericksburg Nationals — Pricing Analysis
This notebook contains all analysis used to evaluate the 2025 transaction data against the 2026 pricing matrix structure.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('./data/FinalTransactionReport.csv')
df['Event Timestamp'] = pd.to_datetime(df['Event Timestamp'], utc=True)
df['Month'] = df['Event Timestamp'].dt.month
df['IsWeekend'] = df['Day of Week'].isin(['Friday', 'Saturday', 'Sunday'])
df['IsAprilMay'] = df['Month'].isin([4, 5])

def assign_tier(row):
    if row['IsAprilMay'] and row['IsWeekend']:
        return 'Weekend Apr/May'
    elif row['IsAprilMay'] and not row['IsWeekend']:
        return 'Weekday Apr/May'
    elif not row['IsAprilMay'] and row['IsWeekend']:
        return 'Weekend Peak'
    else:
        return 'Weekday Peak'

df['Tier'] = df.apply(assign_tier, axis=1)

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())

Shape: (17754, 31)
Columns: ['Event Name', 'Event ID', 'Event Timestamp', 'Venue Name', 'Transaction Timestamp', 'Price Scale', 'Section', 'Row', 'Seats', 'Primary Seat IDs', 'Average Unit Price', 'Sales Total', 'Tickets Sold', 'Market Place', 'Transaction Type', 'Highest Inventory Price', 'Lowest Inventory Price', 'Initial Inventory Price', 'Event Score', 'weather_condition', 'temperature_f', 'precipitation', 'humidity', 'weather_category', 'Days Before Event', 'Start Time', 'Day of Week', 'Month', 'IsWeekend', 'IsAprilMay', 'Tier']


## 1. Price Scale vs Pricing Matrix — Section Mapping

In [2]:
# The pricing matrix uses different section names than the transaction data
# This cell maps them explicitly

section_map = {
    'Club':                        'Club',
    'Club Access - Standing Room':  'Club SRO',
    'Terrace Box Table':            '3rd Base Table/Hi-Tops',
    'Terrace Box High Tops':        '3rd Base Table/Hi-Tops',
    'Diamond Box':                  'Diamond Box',
    'Dugout Box':                   'Dugout Box',
    'Field':                        'Field Seats',
    'SRO':                          'SRO',
}

df['Matrix Section'] = df['Price Scale'].map(section_map)

print('=== PRICE SCALE TO MATRIX SECTION MAPPING ===')
mapping_check = df.groupby(['Price Scale', 'Matrix Section']).size().reset_index(name='Transactions')
print(mapping_check.to_string(index=False))

print('\n=== UNMAPPED PRICE SCALES ===')
print(df[df['Matrix Section'].isna()]['Price Scale'].value_counts())

=== PRICE SCALE TO MATRIX SECTION MAPPING ===
                Price Scale         Matrix Section  Transactions
                       Club                   Club           322
Club Access - Standing Room               Club SRO           318
                Diamond Box            Diamond Box          1476
                 Dugout Box             Dugout Box          8484
                      Field            Field Seats          2281
                        SRO                    SRO          4414
      Terrace Box High Tops 3rd Base Table/Hi-Tops           181
          Terrace Box Table 3rd Base Table/Hi-Tops           278

=== UNMAPPED PRICE SCALES ===
Series([], Name: count, dtype: int64)


## 2. Average Price Paid by Price Scale

In [3]:
# Compare actual prices paid in 2025 to the 2026 matrix prices
# Matrix prices (for reference):
# Club: $77 (all tiers)
# Diamond Box: $24-$28
# Dugout Box: $18-$24
# Field Seats: $14-$18
# SRO: $10

price_by_scale = df.groupby('Price Scale').agg(
    AvgPrice     = ('Average Unit Price', 'mean'),
    MedianPrice  = ('Average Unit Price', 'median'),
    MinPrice     = ('Average Unit Price', 'min'),
    MaxPrice     = ('Average Unit Price', 'max'),
    TotalTickets = ('Tickets Sold', 'sum'),
    Transactions = ('Sales Total', 'count')
).round(2)

print('=== ACTUAL 2025 PRICES BY PRICE SCALE ===')
print(price_by_scale.sort_values('AvgPrice', ascending=False).to_string())

=== ACTUAL 2025 PRICES BY PRICE SCALE ===
                             AvgPrice  MedianPrice  MinPrice  MaxPrice  TotalTickets  Transactions
Price Scale                                                                                       
Club                            76.14         77.0     43.91     101.0           656           322
Club Access - Standing Room     52.23         56.0     43.91      75.0           725           318
Terrace Box High Tops           33.89         31.0     29.96      56.0           719           181
Terrace Box Table               31.73         31.0     10.70      55.0          1100           278
Diamond Box                     25.40         24.0      4.28      50.0          3299          1476
Dugout Box                      21.83         21.0      3.21      46.0         24482          8484
Field                           18.34         18.0      1.00      42.0          6895          2281
SRO                              7.34          8.0      1.00      3

## 3. Price and Volume by Price Scale × Tier

In [4]:
# This is the core analysis — mirrors exactly the structure of the pricing matrix
# Rows = seating sections, Columns = game tiers

tier_scale = df.groupby(['Price Scale', 'Tier']).agg(
    AvgPrice     = ('Average Unit Price', 'mean'),
    Transactions = ('Average Unit Price', 'count'),
    TotalTickets = ('Tickets Sold', 'sum')
).round(2)

print('=== AVG PRICE BY PRICE SCALE AND TIER ===')
print(tier_scale.to_string())

# Pivot for easier reading
price_pivot = df.groupby(['Price Scale', 'Tier'])['Average Unit Price'].mean().round(2).unstack()
col_order = ['Weekday Apr/May', 'Weekday Peak', 'Weekend Apr/May', 'Weekend Peak']
price_pivot = price_pivot[[c for c in col_order if c in price_pivot.columns]]
print('\n=== PRICE PIVOT (Sections × Tiers) ===')
print(price_pivot.to_string())

=== AVG PRICE BY PRICE SCALE AND TIER ===
                                             AvgPrice  Transactions  TotalTickets
Price Scale                 Tier                                                 
Club                        Weekday Apr/May     74.89            35            68
                            Weekday Peak        76.09            71           147
                            Weekend Apr/May     74.97            69           140
                            Weekend Peak        77.01           147           301
Club Access - Standing Room Weekday Apr/May     52.60            39            79
                            Weekday Peak        50.07            50           104
                            Weekend Apr/May     52.43            63           155
                            Weekend Peak        52.72           166           387
Diamond Box                 Weekday Apr/May     24.20           204           433
                            Weekday Peak        24.07   

## 4. Tier-Level Demand Analysis

In [5]:
# How many games per tier, and how many tickets per game?
# Large differences in demand signal whether current tier pricing is appropriate

tier_vol = df.groupby('Tier').agg(
    Games        = ('Event ID', 'nunique'),
    TotalTickets = ('Tickets Sold', 'sum'),
    TotalRevenue = ('Sales Total', 'sum'),
    AvgPrice     = ('Average Unit Price', 'mean')
).round(2)

tier_vol['TicketsPerGame']  = (tier_vol['TotalTickets'] / tier_vol['Games']).round(0)
tier_vol['RevenuePerGame']  = (tier_vol['TotalRevenue'] / tier_vol['Games']).round(2)

print('=== DEMAND BY TIER ===')
print(tier_vol.to_string())

print('\n=== KEY INSIGHT ===')
wknd = tier_vol.loc['Weekend Peak', 'TicketsPerGame']
wkdy = tier_vol.loc['Weekday Peak', 'TicketsPerGame']
print(f'Weekend Peak avg tickets/game:  {wknd:.0f}')
print(f'Weekday Peak avg tickets/game:  {wkdy:.0f}')
print(f'Weekend/Weekday demand ratio:   {wknd/wkdy:.1f}x')

=== DEMAND BY TIER ===
                 Games  TotalTickets  TotalRevenue  AvgPrice  TicketsPerGame  RevenuePerGame
Tier                                                                                        
Weekday Apr/May     12          4416      87794.61     19.47           368.0         7316.22
Weekday Peak        18          6558     111936.68     17.30           364.0         6218.70
Weekend Apr/May     12         12425     258630.87     20.67          1035.0        21552.57
Weekend Peak        24         24814     509230.25     20.34          1034.0        21217.93

=== KEY INSIGHT ===
Weekend Peak avg tickets/game:  1034
Weekday Peak avg tickets/game:  364
Weekend/Weekday demand ratio:   2.8x


## 5. April/May Discount — Is It Justified?

In [6]:
# The pricing matrix discounts April/May games
# Does the data support this? Compare tickets sold per game by month

monthly = df.groupby('Month').agg(
    Games        = ('Event ID', 'nunique'),
    TotalTickets = ('Tickets Sold', 'sum'),
    TotalRevenue = ('Sales Total', 'sum'),
    AvgPrice     = ('Average Unit Price', 'mean')
).round(2)
monthly['TicketsPerGame']  = (monthly['TotalTickets'] / monthly['Games']).round(0)
monthly['RevenuePerGame']  = (monthly['TotalRevenue'] / monthly['Games']).round(2)

month_names = {4:'April', 5:'May', 6:'June', 7:'July', 8:'August', 9:'September'}
monthly.index = monthly.index.map(month_names)

print('=== TICKETS AND REVENUE PER GAME BY MONTH ===')
print(monthly[['Games','TicketsPerGame','RevenuePerGame','AvgPrice']].to_string())

# Also compare Apr/May vs rest by weekend/weekday
print('\n=== AVG PRICE: APRIL/MAY vs REST × WEEKEND/WEEKDAY ===')
summary = df.groupby(['IsAprilMay', 'IsWeekend'])['Average Unit Price'].mean().round(2)
summary.index = summary.index.map(lambda x: f"{'Apr/May' if x[0] else 'Jun-Sep'} {'Weekend' if x[1] else 'Weekday'}")
print(summary.to_string())

=== TICKETS AND REVENUE PER GAME BY MONTH ===
           Games  TicketsPerGame  RevenuePerGame  AvgPrice
Month                                                     
April         12           619.0        12993.89     20.86
May           12           784.0        15874.90     19.86
June          12           650.0        12374.16     18.76
July          12           987.0        20328.53     20.65
August        12           696.0        13370.32     18.96
September      6           563.0        11381.81     20.07

=== AVG PRICE: APRIL/MAY vs REST × WEEKEND/WEEKDAY ===
Jun-Sep Weekday    17.30
Jun-Sep Weekend    20.34
Apr/May Weekday    19.47
Apr/May Weekend    20.67


## 6. Advance Purchase Pricing (Floor vs Advanced)

In [7]:
# The pricing matrix has a 'Floor' and 'Advanced' price per section
# This analyzes whether actual purchase timing affects prices paid
# and validates whether the advance/floor threshold is set appropriately

bins   = [0, 1, 7, 14, 30, 60, 999]
labels = ['Game Day', '2-7 days', '8-14 days', '15-30 days', '31-60 days', '60+ days']
df['PurchaseTiming'] = pd.cut(df['Days Before Event'], bins=bins, labels=labels, right=True)

timing_summary = df.groupby('PurchaseTiming').agg(
    AvgPrice     = ('Average Unit Price', 'mean'),
    Transactions = ('Average Unit Price', 'count'),
    TotalTickets = ('Tickets Sold', 'sum')
).round(2)
timing_summary['TicketShare'] = (timing_summary['TotalTickets'] / timing_summary['TotalTickets'].sum() * 100).round(1)

print('=== PRICE AND VOLUME BY PURCHASE TIMING ===')
print(timing_summary.to_string())

# Also look at this per price scale
print('\n=== AVG PRICE BY TIMING × PRICE SCALE ===')
timing_scale = df.groupby(['PurchaseTiming', 'Price Scale'])['Average Unit Price'].mean().round(2).unstack()
print(timing_scale.to_string())

=== PRICE AND VOLUME BY PURCHASE TIMING ===
                AvgPrice  Transactions  TotalTickets  TicketShare
PurchaseTiming                                                   
Game Day           20.62          1746          4907         17.0
2-7 days           22.44          3977         11836         40.9
8-14 days          23.36          1351          4230         14.6
15-30 days         23.59          1128          3631         12.5
31-60 days         24.51           793          2556          8.8
60+ days           26.96           584          1778          6.1

=== AVG PRICE BY TIMING × PRICE SCALE ===
Price Scale      Club  Club Access - Standing Room  Diamond Box  Dugout Box  Field    SRO  Terrace Box High Tops  Terrace Box Table
PurchaseTiming                                                                                                                     
Game Day        76.32                        49.65        25.79       22.44  17.56   7.44                  33.56         

/tmp/ipykernel_7014/337289191.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  timing_summary = df.groupby('PurchaseTiming').agg(
/tmp/ipykernel_7014/337289191.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  timing_scale = df.groupby(['PurchaseTiming', 'Price Scale'])['Average Unit Price'].mean().round(2).unstack()


## 7. Opponent Impact on Demand

In [8]:
# The current pricing matrix does not vary by opponent
# This analyzes whether opponent affects ticket volume and price paid

opponent_summary = df.groupby('Event Name').agg(
    Games            = ('Event ID', 'nunique'),
    TotalTickets     = ('Tickets Sold', 'sum'),
    TotalRevenue     = ('Sales Total', 'sum'),
    AvgPrice         = ('Average Unit Price', 'mean')
).round(2)
opponent_summary['TicketsPerGame']  = (opponent_summary['TotalTickets'] / opponent_summary['Games']).round(0)
opponent_summary['RevenuePerGame']  = (opponent_summary['TotalRevenue'] / opponent_summary['Games']).round(2)

print('=== DEMAND BY OPPONENT ===')
print(opponent_summary[['Games','TicketsPerGame','RevenuePerGame','AvgPrice']]
      .sort_values('TicketsPerGame', ascending=False).to_string())

# Opponent vs day-of-week interaction — is Delmarva just scheduled on better days?
print('\n=== OPPONENT × DAY OF WEEK (tickets per game) ===')
opp_day = df.groupby(['Event Name', 'Day of Week']).agg(
    Games        = ('Event ID', 'nunique'),
    TotalTickets = ('Tickets Sold', 'sum')
)
opp_day['TicketsPerGame'] = (opp_day['TotalTickets'] / opp_day['Games']).round(0)
print(opp_day['TicketsPerGame'].unstack().fillna('-').to_string())

=== DEMAND BY OPPONENT ===
                           Games  TicketsPerGame  RevenuePerGame  AvgPrice
Event Name                                                                
Delmarva Shorebirds            9          1046.0        22798.53     21.56
Lynchburg Hillcats             9           760.0        16036.63     20.87
Carolina Mudcats              12           722.0        14134.47     19.28
Fayetteville Woodpeckers      12           663.0        11373.33     17.27
Salem Red Sox                 12           657.0        13234.81     19.87
Charleston Riverdogs           6           638.0        13620.36     21.37
Kannapolis Cannon Ballers      6           606.0        11907.09     19.54

=== OPPONENT × DAY OF WEEK (tickets per game) ===
Day of Week                Friday  Saturday  Sunday  Thursday  Tuesday  Wednesday
Event Name                                                                       
Carolina Mudcats           1218.0     994.0  1166.0     489.0    302.0      160.0
C

## 8. Per-Game Ticket Totals — Identifying Peak Games

In [9]:
# The pricing matrix has a 'Peak' tier with no definition in the data
# This identifies what characteristics the highest-demand games share

game_summary = df.groupby(['Event ID', 'Event Name', 'Day of Week', 'Start Time', 'Month']).agg(
    TotalTickets = ('Tickets Sold', 'sum'),
    TotalRevenue = ('Sales Total', 'sum'),
    AvgUnitPrice = ('Average Unit Price', 'mean')
).reset_index().round(2)

print(f'Ticket range: {game_summary["TotalTickets"].min()} – {game_summary["TotalTickets"].max()}')
print(f'Mean:   {game_summary["TotalTickets"].mean():.0f}')
print(f'Median: {game_summary["TotalTickets"].median():.0f}')

# Top 10 and bottom 10 games
print('\n=== TOP 10 GAMES BY TICKETS SOLD ===')
print(game_summary.nlargest(10, 'TotalTickets')
      [['Event Name','Day of Week','Start Time','Month','TotalTickets','TotalRevenue']].to_string(index=False))

print('\n=== BOTTOM 10 GAMES BY TICKETS SOLD ===')
print(game_summary.nsmallest(10, 'TotalTickets')
      [['Event Name','Day of Week','Start Time','Month','TotalTickets','TotalRevenue']].to_string(index=False))

# Define 'Peak' as top quartile in tickets sold
top_quartile = game_summary['TotalTickets'].quantile(0.75)
print(f'\nTop quartile threshold (potential Peak definition): {top_quartile:.0f} tickets')
peak_games = game_summary[game_summary['TotalTickets'] >= top_quartile]
print('\nPeak game breakdown by day of week:')
print(peak_games['Day of Week'].value_counts())
print('\nPeak game breakdown by month:')
print(peak_games['Month'].value_counts().sort_index())

Ticket range: 94 – 2987
Mean:   730
Median: 634

=== TOP 10 GAMES BY TICKETS SOLD ===
               Event Name Day of Week Start Time  Month  TotalTickets  TotalRevenue
      Delmarva Shorebirds      Friday    7:05 PM      7          2987      82542.85
            Salem Red Sox    Saturday    7:05 PM      8          2092      43491.85
      Delmarva Shorebirds    Saturday    7:05 PM      7          1770      34847.37
 Fayetteville Woodpeckers    Saturday    7:05 PM      8          1726      31185.55
         Carolina Mudcats      Sunday    1:35 PM      5          1624      31173.95
       Lynchburg Hillcats    Saturday    7:05 PM      5          1444      31235.40
         Carolina Mudcats    Saturday    7:05 PM      5          1354      27453.72
Kannapolis Cannon Ballers    Saturday    7:05 PM      6          1335      26214.45
         Carolina Mudcats      Friday    7:05 PM      9          1264      25880.63
 Fayetteville Woodpeckers   Wednesday    6:35 PM      7          1262     

## 9. Weather Impact on Attendance

In [10]:
# Does weather affect tickets sold per game?

weather_game = df.groupby(['Event ID', 'weather_category', 'temperature_f']).agg(
    TicketsSold = ('Tickets Sold', 'sum')
).reset_index()

print('=== TICKETS PER GAME BY WEATHER CATEGORY ===')
print(weather_game.groupby('weather_category')['TicketsSold']
      .agg(['mean','median','min','max','count']).round(1).to_string())

# Temperature bins
weather_game['TempBin'] = pd.cut(
    weather_game['temperature_f'],
    bins=[0, 55, 65, 75, 85, 120],
    labels=['<55°F', '55-65°F', '65-75°F', '75-85°F', '>85°F']
)

print('\n=== TICKETS PER GAME BY TEMPERATURE ===')
print(weather_game.groupby('TempBin')['TicketsSold']
      .agg(['mean','median','count']).round(1).to_string())

=== TICKETS PER GAME BY WEATHER CATEGORY ===
                   mean  median  min   max  count
weather_category                                 
Clear             788.0   717.0  165  2987     26
Cloudy            691.7   590.0   94  1770     29
Rainy             696.9   499.0  215  2092     11

=== TICKETS PER GAME BY TEMPERATURE ===
          mean  median  count
TempBin                      
<55°F    544.8   417.5      4
55-65°F  775.4   809.0      7
65-75°F  597.9   332.0     16
75-85°F  860.2   674.5     28
>85°F    632.1   567.0     11


/tmp/ipykernel_7014/895855103.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(weather_game.groupby('TempBin')['TicketsSold']


## 10. Dynamic Pricing Signal — Initial vs Final Price

In [11]:
# Initial Inventory Price = price when ticket was first listed
# Average Unit Price = price actually paid
# Difference reveals whether dynamic pricing is occurring and in which direction

df_valid = df.dropna(subset=['Initial Inventory Price'])
df_valid = df_valid[df_valid['Initial Inventory Price'] > 0].copy()
df_valid['PriceChange']    = df_valid['Average Unit Price'] - df_valid['Initial Inventory Price']
df_valid['PriceChangePct'] = (df_valid['PriceChange'] / df_valid['Initial Inventory Price']) * 100

print(f'Rows with Initial Price data: {len(df_valid):,}')
print('\n=== PRICE CHANGE DISTRIBUTION ===')
print(df_valid['PriceChangePct'].describe().round(2))

print('\n=== PRICE CHANGE BY PURCHASE TIMING ===')
bins   = [0, 1, 7, 14, 30, 60, 999]
labels = ['Game Day', '2-7 days', '8-14 days', '15-30 days', '31-60 days', '60+ days']
df_valid['PurchaseTiming'] = pd.cut(df_valid['Days Before Event'], bins=bins, labels=labels)
print(df_valid.groupby('PurchaseTiming')['PriceChangePct'].mean().round(2).to_string())

print('\n=== PRICE CHANGE BY PRICE SCALE ===')
print(df_valid.groupby('Price Scale')['PriceChangePct'].mean().round(2)
      .sort_values(ascending=False).to_string())

Rows with Initial Price data: 15,295

=== PRICE CHANGE DISTRIBUTION ===
count    15295.00
mean        -9.74
std         29.33
min        -94.44
25%        -16.67
50%         -4.76
75%          4.17
max        216.67
Name: PriceChangePct, dtype: float64

=== PRICE CHANGE BY PURCHASE TIMING ===
PurchaseTiming
Game Day     -8.14
2-7 days     -1.81
8-14 days     2.38
15-30 days    1.50
31-60 days    1.16
60+ days     -1.61

=== PRICE CHANGE BY PRICE SCALE ===
Price Scale
Dugout Box                      3.10
Terrace Box High Tops           3.01
Field                           0.93
Terrace Box Table               0.35
Diamond Box                    -1.60
Club                           -3.49
Club Access - Standing Room    -5.60
SRO                           -38.94


/tmp/ipykernel_7014/2184219614.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df_valid.groupby('PurchaseTiming')['PriceChangePct'].mean().round(2).to_string())
